In [ ]:
import {tensor, torus} from '@torus';
import {nn} from '@torus/neuro-module';
import {webgpu} from '@torus/web-gpu.js';
tensor.LEARNING_RATE = .1
torus.manual_seed(22)

In [ ]:
BinNet = class BinNet extends EventTarget {
    static max = 2 ** 32 - 1;
    constructor(){
        super(...arguments)
    }
    get gpu(){
        return BinNet.gpu;
    }
    static vec2bits(vector = new Uint32Array(), split = 0){
        if(split){
            let list = [];
            for(let i = 0; i<vector.length; i += split){
                list.push(vector.subarray(i , i  + split));
            }
            return this.printW(list)
        }
        return Array.prototype.map.call(vector, v=>v.toString(2).padStart(32, '0')).join(' ')
        
    }
    static printW(weights){
        return weights.map(w=>Array.prototype.map.call(w, v=>v.toString(2).padStart(32, '0')).join(''))//.join('');
    }    
     
    static create_random_vector(size){
        return this.create_zeros_vector(size).map(v => Math.trunc(this.max32 * Math.random()));
    }
    static create_zeros_vector(size){
        return new Uint32Array(size);
    }
    static create_ones_vector(size){
        return this.create_zeros_vector(size).fill(this.max);
    }
    static hamming_distance(a, b) {
        return a.reduce((sum, v1, i) => sum + this.popcount32(v1 ^ b[i]), 0);
    } 
    static bitSimilarityUint32(vec1, vec2) {
      // 1. Проверяем равенство длин массивов
      if (vec1.length !== vec2.length) return 0;
      if (vec1.length === 0) return 100;
    
      let totalDiffBits = 0;
      const len = vec1.length;
    
      for (let i = 0; i < len; i++) {
        // Оператор XOR (^) дает 1 только там, где биты РАЗЛИЧАЮТСЯ
        let xor = vec1[i] ^ vec2[i];
        
        // Алгоритм Брайана Кернигана для быстрого подсчета единиц

        while (xor !== 0) {
          totalDiffBits++;
          xor &= (xor - 1); // Сбрасывает самый младший установленный бит
        }
      }
    
      const totalBits = len * 32;
      const matchingBits = totalBits - totalDiffBits;
    
      return 1 - matchingBits / totalBits;
    }
 
    static get gpu(){
        
        return webgpu;
    }    
}

In [ ]:
BinLayer = class BinLayer extends BinNet{
    _batchSize = 0;
    _shaders = {};
    rate = 3;
    constructor(config = {}){
        super(...arguments);
        this.in_size = config.in_size || 1;
        this.out_size = config.out_size || 1;
        this.divider = config.divider || 1;
        this.id = config.id || 0;
        if(this.in_size % this.divider)
            throw new Error(`Размерность входа слоя "${this.id}" in_size=${this.in_size}, что не соответствует кратности делителя divider=${this.divider}!`);
        if(this.out_size % this.divider)
            throw new Error(`Размерность выхода слоя "${this.id}" out_size=${this.out_size}, что не соответствует кратности делителя divider=${this.divider}!`);
        
        this.in_dim = this.in_size * 32;
        this.out_dim = this.out_size * 32;
        this.all_weights_size = this.in_size / this.divider * this.out_size * 32;
        this.weight_size = this.all_weights_size / this.out_dim;
        this.weights = null;
    }
    init(data){
        this.weights = data || BinNet.create_ones_vector(this.all_weights_size);
        this.gpu.writeData(this.weights, false, 'weights: ' + this.id);
    }
    get batchSize(){
        return this._batchSize;
    }
    set batchSize(n){
        if(this._batchSize === n)
            return;
        this._batchSize = n;
        this.input = new Uint32Array(n * this.in_size);
        this.target = new Uint32Array(n * this.out_size);
    }
    forward(input){
        if(Array.isArray(input)){
            this.batchSize = input.length;
            input = input.reduce((res, v, i)=>{
                res.set(v, i * this.in_size);
                return res;
            }, this.input);
            this.gpu.writeData(this.input, true, 'input: ' + this.id + ', bs='+this.batchSize);
        }
        if(!this.input){
            this.input = input;
            this.gpu.writeData(this.input, false, 'input: ' + this.id);
        }
        else if(this.input !== input){
            this.input.set(input);
            this.gpu.writeData(this.input, true, 'input: ' + this.id);
        }
        if(!this._shaders.FORWARD){
            let wg = this.gpu.compute_info(this.out_size);
            this._shaders.FORWARD = {count: wg.count};
            this.output = BinNet.create_zeros_vector(this.out_size);
            this.gpu.writeData(this.output, false, 'output: ' + this.id);
            
            // Вычисляем точные шаги подсетей на стороне JS (в штуках u32-блоков)
            const blocksPerSubnetOut = this.out_size / this.divider;
            // Количество блоков входа на одну подсеть (это и есть реальный w_size для шейдера)
            const blocksPerSubnetIn = this.weight_size; 
        
            let code = [
                `// FORWARD`,
                `@group(0) @binding(0) var<storage, read> inputs: array<u32>;`,
                `@group(0) @binding(1) var<storage, read> weights: array<u32>;`,
                `@group(0) @binding(2) var<storage, read_write> outputs: array<u32>;`,
                `@compute @workgroup_size(${wg.size})`,
                `fn main(@builtin(global_invocation_id) id: vec3<u32>) {`,
                wg.code, // "let idx = id.x;" (индекс выходного блока u32)
                `    const w_size = ${blocksPerSubnetIn}u;`,
                `    var out_value = 0u;`, 
                `    let out_dim_start = idx * 32u;`,                
                
                // Номер группы (подсети) считаем ОДИН раз на поток через индекс выходного блока u32
        this.divider > 1 ? `    let subnet_id = idx / ${blocksPerSubnetOut}u;` : `    let subnet_id = 0u;`,
                // Смещение по входу: номер подсети * количество входных блоков на подсеть
                `    let input_start = subnet_id * w_size;`,
        
                `    for(var o = 0u; o < 32u; o++){`,
                `        let out_idx = o + out_dim_start;`,
                `        let start = out_idx * w_size;`,
                `        var sum: i32 = 0;`, 
        
                `        for(var i = 0u; i < w_size; i++) {`,
                `            let weight = weights[start + i];`,
                `            let input = inputs[input_start + i];`, 
                `            sum += i32(countOneBits(input & weight)) - i32(countOneBits(input & ~weight));`,                                
                `        }`,  
                `        if(sum > 0) { out_value = out_value | (1u << o); }`,
                `    }`,     
                `    outputs[idx] = out_value;`,
                `}`
            ].flat(Infinity).join('\n');
            
            this._shaders.FORWARD.shader = this.gpu.compile(code);
        }

        this.gpu.compute(this._shaders.FORWARD.shader, [this.input, this.weights, this.output], this._shaders.FORWARD.count);
        return this.output;  
    }
    back(target){
        if(Array.isArray(target)){
            if(this.batchSize !== target.length)
                throw new Error('this.batchSize !== target.length');
            target = target.reduce((res, v, i)=>{
                res.set(v, i * this.out_size);
                return res;
            }, this.target);
            this.gpu.writeData(this.target, true, 'target: ' + this.id + ', bs='+this.batchSize);
        }
        if(!this.target){
            this.target = target;
            this.gpu.writeData(this.target, false, 'target: ' + this.id);
        }        
        else if(this.target !== target){
            this.target.set(target);
            this.gpu.writeData(this.target, true, 'target: ' + this.id);
        }        
        this._update();
        if(!this.id)
            return;
        if(!this._shaders.BACK){
            let wg = this.gpu.compute_info(this.in_size); 
            this._shaders.BACK = {
                count: wg.count,
                target: BinNet.create_zeros_vector(this.in_size)
            };
            this.gpu.writeData(this._shaders.BACK.target, false, 'back_target: ' + this.id);
            
            const blocksPerSubnetIn = this.in_size / this.divider;
            const blocksPerSubnetOut = this.out_size / this.divider;
            const neuronsPerSubnetOut = blocksPerSubnetOut * 32;
        
            // Вычисляем, сколько бит нужно для хранения максимальной суммы (log2 от количества нейронов в подсети)
            // Например, для 32 нейронов нужно 6 бит (знак + 5 бит на число 32)
            const numBitsRequired = Math.ceil(Math.log2(neuronsPerSubnetOut + 1)) + 1;
        
            let code = [
                `// ULTRA-FAST SWAR BIT-PARALLEL TRANSPOSITION`,
                `@group(0) @binding(0) var<storage, read> inputs: array<u32>;`,       
                `@group(0) @binding(1) var<storage, read> weights: array<u32>;`,      
                `@group(0) @binding(2) var<storage, read_write> outputs: array<u32>;`, 
                `@compute @workgroup_size(${wg.size})`,
                `fn main(@builtin(global_invocation_id) id: vec3<u32>) {`,
                wg.code, 
                `    if (idx >= ${this.in_size}u) { return; }`,
                
                `    const w_size = ${this.weight_size}u;`, 
                
        this.divider > 1 ? `    let subnet_id = idx / ${blocksPerSubnetIn}u;` : `    let subnet_id = 0u;`,
        this.divider > 1 ? `    let in_block_offset = idx % ${blocksPerSubnetIn}u;` : `    let in_block_offset = idx;`,
                
                `    let r_start = subnet_id * ${neuronsPerSubnetOut}u;`,
                `    let r_end = r_start + ${neuronsPerSubnetOut}u;`,
                
                // Вместо 32 счетчиков i32, мы создаем N регистров-слоев u32.
                // Каждый регистр хранит k-ый бит счетчика для всех 32 колонок сразу!
                new Array(numBitsRequired).fill(0).map((_, i) => `    var c${i} = 0u;`).join('\n'),
                
                `    for(var r = r_start; r < r_end; r++) {`,
                `        let weight_word_idx = (r * w_size) + in_block_offset;`,
                `        let weight_word = weights[weight_word_idx];`,
                
                `        let input_word_idx = r / 32u;`,
                `        let input_bit_shift = r % 32u;`,
                `        let input_bit = (inputs[input_word_idx] >> input_bit_shift) & 1u;`,
                
                // Размножаем бит цели на маску 0x00000000 или 0xFFFFFFFF
                `        let input_mask = 0u - input_bit;`,
                
                // Идеальное совпадение: 1 во всех битах, где веса совпали с целью (+1 к сумме), 0 где не совпали (-1)
                `        let matches = ~(weight_word ^ input_mask);`,
                
                // Многоразрядный параллельный сумматор (SWAR Adder)
                // Мы прибавляем 1 к счетчику, если matches имеет 1, и вычитаем 1 (прибавляем -1, то есть инверсию), если 0.
                // Для простоты: смещаем баланс. За совпадение делаем +2, за несовпадение +0 (просто прибавляем matches дважды)
                // В конце мажоритарный порог сместится: сумма должна быть > neuronsPerSubnetOut
                `        var carry = matches;`,
                // Первый проход сложения (+1)
                new Array(numBitsRequired).fill(0).map((_, i) => {
                    if (i === numBitsRequired - 1) return `        c${i} = c${i} ^ carry;`;
                    return `        let next_carry${i} = c${i} & carry;\n        c${i} = c${i} ^ carry;\n        carry = next_carry${i};`;
                }).join('\n'),
                
                // Второй проход сложения (+1), суммарно получили +2 за каждое совпадение
                `        carry = matches;`,
                new Array(numBitsRequired).fill(0).map((_, i) => {
                    if (i === numBitsRequired - 1) return `        c${i} = c${i} ^ carry;`;
                    return `        let next_carry_b${i} = c${i} & carry;\n        c${i} = c${i} ^ carry;\n        carry = next_carry_b${i};`;
                }).join('\n'),
                
                `    }`,
                
                // Каждое совпадение дало +2. Базовый сдвиг от отсутствия совпадений равен 0.
                // Чтобы итоговый sum был > 0 (совпадений больше половины), накопленное значение 
                // в каждом битовом канале должно быть строго больше, чем neuronsPerSubnetOut.
                // Вычитаем пороговое значение из нашего распределенного счетчика c0...cN
                `    var threshold = ${neuronsPerSubnetOut}u;`,
                `    var out_value = 0u;`,
                
                // Извлекаем финальный знак (> threshold) для каждого из 32 битов через развернутый побитовый сдвиг
                new Array(32).fill(0).map((_, b) => {
                    // Собираем значение счетчика для конкретного бита 'b' из слоев c0...cN
                    const valTerms = new Array(numBitsRequired).fill(0).map((_, i) => `(((c${i} >> ${b}u) & 1u) << ${i}u)`).join(' | ');
                    return `    if ((${valTerms}) > threshold) { out_value = out_value | (1u << ${b}u); }`;
                }).join('\n'),
                
                `    outputs[idx] = out_value;`,
                `}`
            ].flat(Infinity).join('\n');
                    
            this._shaders.BACK.shader = this.gpu.compile(code);
        }

            
    
        this.gpu.compute(this._shaders.BACK.shader, [
            this.target, 
            this.weights, 
            this._shaders.BACK.target
        ], this._shaders.BACK.count);
        
        return this._shaders.BACK.target;
    }
    train(input, target){
        this.forward(input);
        this.back(target);
    }
    _update(){
        
        
        if(!this._shaders.ERROR){    
            let wg = this.gpu.compute_info(this.out_size);
            this._shaders.ERROR = {
                count: wg.count,
                errors: new Float32Array(this.out_size).fill(1),
            }  
            this.gpu.writeData(this._shaders.ERROR.errors, true, 'error: ' + this.id);
            let code = [
                `// ERROR`,
                `@group(0) @binding(0) var<storage, read> outputs: array<u32>;`,
                `@group(0) @binding(1) var<storage, read> targets: array<u32>;`,
                `@group(0) @binding(2) var<storage, read_write> errors:  array<f32>;`,                
                `@compute @workgroup_size(${wg.size})`,
                `fn main(@builtin(global_invocation_id) id: vec3<u32>) {`,
                wg.code,                
                `    var error = outputs[idx] ^ targets[idx];`,
                `    error = countOneBits(error);`,                
                `    errors[idx] = f32(error) / 32.0;`,
                `}`].flat(Infinity).join('\n');
            this._shaders.ERROR.shader = this.gpu.compile(code);              
        }
        
        
        this.gpu.compute(this._shaders.ERROR.shader, [
            this.output, 
            this.target, 
            this._shaders.ERROR.errors
        ], this._shaders.ERROR.count);  
        
        
        
        
        
        if(!this._shaders.UPDATE){
            // Теперь создаем потоки строго поблочно (по количеству u32 блоков на выходе)
            let wg = this.gpu.compute_info(this.out_size);
            this._shaders.UPDATE = {count: wg.count};
            
            // Количество u32-блоков выходов в одной подсети
            const blocksPerSubnetOut = this.out_size / this.divider;
        
            let code = [
                `// FIXED STOCHASTIC HEBB UPDATE (BLOCK-WISE)`,
                `fn xorshift32(state: ptr<function, u32>) -> u32 {`,
                `    var x = *state;`,
                `    x ^= x << 13u;`,
                `    x ^= x >> 17u;`,
                `    x ^= x << 5u;`,
                `    return x;`,
                `}`,
                `@group(0) @binding(0) var<storage, read> inputs: array<u32>;`,
                `@group(0) @binding(1) var<storage, read_write> weights: array<u32>;`,
                `@group(0) @binding(2) var<storage, read> targets: array<u32>;`,
                `@group(0) @binding(3) var<storage, read> errors: array<f32>;`,
                `@group(0) @binding(4) var<uniform> seed: u32;`,
                `@compute @workgroup_size(${wg.size})`,
                `fn main(@builtin(global_invocation_id) id: vec3<u32>) {`,
                wg.code, // Здесь генерируется "let idx = id.x;" (индекс выходного блока u32)
                `    if (idx >= ${this.out_size}u) { return; }`,
                
                `    const w_size = ${this.weight_size}u;`,
                `    let error = errors[idx];`,   
                `    let target_word = targets[idx];`,
                
                // Вычисляем номер подсети и стартовый индекс во входах один раз на весь блок
        this.divider > 1 ? `    let subnet_id = idx / ${blocksPerSubnetOut}u;` : `    let subnet_id = 0u;`,
                `    let input_start = subnet_id * w_size;`,
            
                `    var rnd: u32 = idx ^ seed;`,
                // Считаем pre_mask один раз на весь блок u32
                `    let loop_count = i32(clamp(2.0 / (0.14 + error), 1.0, 11.0));`, 
                `    var pre_mask = 0xFFFFFFFFu;`,
                `    for(var r = 0; r < loop_count; r++){`,
                `        rnd = xorshift32(&rnd);`,                
                `        pre_mask = pre_mask & rnd;`,
                `    }`,
                // Внешний цикл: идем по 32 нейронам (битам) текущего блока выходов
                `    for(var o = 0u; o < 32u; o++) {`,
                // Глобальный порядковый номер текущего нейрона в сети (заменяет старый idx)
                `        let neuron_id = (idx * 32u) + o;`,
                `        let start = neuron_id * w_size;`,
                // Извлекаем целевой бит для конкретного o-го нейрона
                `        let target_bit = (target_word >> o) & 1u;`,
                // Внутренний цикл: обновляем блоки весов для этого конкретного нейрона
                `        for(var i = 0u; i < w_size; i++) {`,
                `            let weight = weights[start + i];`,
                `            var input = inputs[input_start + i];`, 
                `            if (target_bit == 0u) { input = ~input; }`,                
                `            rnd = xorshift32(&rnd);`,                
                `            let mask = rnd & pre_mask;`,                
                // Обновляем глобальную матрицу весов
                `            weights[start + i] = (weight & ~mask) | (input & mask);`,             
                `        }`,
                `    }`,                
                `}`].flat(Infinity).join('\n');    
                
            this._shaders.UPDATE.shader = this.gpu.compile(code); 
            this._shaders.UPDATE.seed = new Uint32Array(1);
        }
       

        this._shaders.UPDATE.seed[0] = Math.random() * BinNet.max;        
        this.gpu.writeData(this._shaders.UPDATE.seed, true, 'seed: '+ this.id, {usage: GPUBufferUsage.UNIFORM  | GPUBufferUsage.COPY_DST});

        this.gpu.compute(this._shaders.UPDATE.shader, [
            this.input, 
            this.weights, 
            this.target, 
            this._shaders.ERROR.errors,           
            this._shaders.UPDATE.seed,
        ], this._shaders.UPDATE.count);        
    }
 
    get model(){
        
    }
    get param_count(){
        return this.weights.length * 32;
    }
    get errors(){
        if(this._shaders.ERROR?.errors)
            return this.gpu.readData(this._shaders.ERROR.errors);
        return []
    }
    get error(){
        return this.errors.then?.(errors => {
            let res = errors.reduce((res, v)=>res + v, 0.0);
            res = res / errors.length;
            return res
        }) || 1;
    }    
    get bin_weights(){
        return this.gpu.readData(this.weights).then(weights=>{
            let result = []
            for(let w = 0; w<weights.length; w += this.weight_size){
                let sub = weights.subarray(w, w + this.weight_size);
                sub = Array.prototype.map.call(sub, v=>v.toString(2).padStart(32, '0')).join(' ');
                
                
                if(w / this.weight_size % 32 === 0){
                    let text = ('out block: '+ Math.trunc(w / this.weight_size / 32) + ' of ' + this.id + ' layer').padEnd(this.weight_size * 32 + this.weight_size - 1, '_')
                    result.push(text)
                }
                    
                result.push(sub)
            }
            return result;
        })
    }
}
BinNet.gpu.clear();
layer = new BinLayer({in_size: 4, out_size: 4, divider: 1});
>layer.weight_size;
input = BinNet.create_random_vector(layer.in_size);
target = BinNet.create_random_vector(layer.out_size);


<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>layer.weight_size</label>

4

In [ ]:
learn_count = 100;
let percent = 0;
for(let i = 0 ; i<learn_count; i++){
    result = layer.forward(input);
    layer.back(target);
//     let n_percent = Math.round(i / learn_count * 100);
//     if(percent === n_percent)
//         continue;
//     percent = n_percent
//     await this.setProgress(percent);
}
result = await BinNet.gpu.readData(result);

>BinNet.vec2bits(target);
>BinNet.vec2bits(result);
>error = await layer.error;
w = await layer.bin_weights;
>w

In [ ]:
class BinaryEmbeddingLayer {
    constructor(vocabSize, embeddingSizeBlocks, webGpuInstance) {
        this.vocabSize = vocabSize;                  // Размер словаря (например, 65536)
        this.embSize = embeddingSizeBlocks;          // Сколько u32-блоков занимает один токен (in_size первого слоя)
        this.gpu = webGpuInstance;
        
        // Таблица эмбеддингов: для каждого токена свой бинарный вектор
        this.weights = new Uint32Array(this.vocabSize * this.embSize);
        // Заполняем случайными битами для инициализации
        for(let i = 0; i < this.weights.length; i++) {
            this.weights[i] = Math.floor(Math.random() * 0xFFFFFFFF);
        }
        
        this.gpu.writeData(this.weights, false, 'embedding_table');
        this._shader = null;
    }

    // Принимает Uint32Array с ID токенов (например, от js-tiktoken)
    // Возвращает бинарный вектор (массив u32 блоков), готовый для forward-прохода сети
    getEmbeddings(tokenIds) {
        const numTokens = tokenIds.length;
        const totalOutputBlocks = numTokens * this.embSize;
        
        // Создаем или забираем буфер вывода на GPU
        const outputBuffer = this.gpu.createBuffer(totalOutputBlocks * 4, false, 
            GPUBufferUsage.STORAGE | GPUBufferUsage.COPY_SRC | GPUBufferUsage.COPY_DST, 'embedded_tokens');
        
        // Загружаем ID токенов на GPU
        const tokensGpuBuffer = this.gpu.writeData(tokenIds, true, 'input_token_ids');

        if (!this._shader) {
            // Конфигурируем рабочие группы по количеству ТОКЕНОВ на входе
            let wg = this.gpu.compute_info(numTokens);
            this._shader = { count: wg.count, size: wg.size };

            let code = [
                `// BINARY EMBEDDING LOOKUP`,
                `@group(0) @binding(0) var<storage, read> token_ids: array<u32>;`,       // Массив ID токенов [id1, id2, id3...]
                `@group(0) @binding(1) var<storage, read> emb_table: array<u32>;`,       // Таблица эмбеддингов (vocab_size * emb_size)
                `@group(0) @binding(2) var<storage, read_write> outputs: array<u32>;`,   // Итоговый плоский бинарный вектор
                `@compute @workgroup_size(${wg.size})`,
                `fn main(@builtin(global_invocation_id) id: vec3<u32>) {`,
                wg.code, // idx — это индекс ТОКЕНА в текущем тексте (от 0 до numTokens - 1)
                `    if (idx >= ${numTokens}u) { return; }`,
                
                `    const emb_size = ${this.embSize}u;`,
                `    let token_id = token_ids[idx];`,
                
                // Находим, где в таблице эмбеддингов начинается строка этого токена
                `    let table_start = token_id * emb_size;`,
                // Находим, куда в выходном буфере мы должны скопировать эту строку
                `    let output_start = idx * emb_size;`,
                
                // Копируем всю u32-строку эмбеддинга для данного токена
                `    for(var i = 0u; i < emb_size; i++) {`,
                `        outputs[output_start + i] = emb_table[table_start + i];`,
                `    }`,
                `}`
            ].flat(Infinity).join('\n');

            this._shader.module = this.gpu.compile(code);
        }

        // Запускаем шейдер сборки эмбеддингов
        this.gpu.compute(
            this._shader.module, 
            [tokensGpuBuffer, this.weights, outputBuffer], 
            this._shader.count
        );

        return outputBuffer; // Возвращаем GPUBuffer, который сразу скармливаем в forward() первого слоя
    }
}


In [ ]:
BinNetEntropyTokenizer = class BinNetEntropyTokenizer {
    constructor(targetVocabSize = 256) {
        this.targetVocabSize = targetVocabSize;
        this.vocab = new Map();
        this.inverseVocab = new Map();
        this.merges = new Map();
        this.initBasic();
        this.textEncoder = new TextEncoder();
        this.textDecoder = new TextDecoder();
    }

    // Инициализируем базовую таблицу байт (0-255)
    initBasic() {
        for (let i = 0; i < 256; i++) {
            // Ключом в Map делаем строку из байта, чтобы избежать проблем со ссылками на массивы
            this.vocab.set(i.toString(), i);
            this.inverseVocab.set(i, new Array().concat(i));
        }
    }

    // Обучение словаря на основе максимизации информационного выигрыша
    train(text, targetSize) {
        this.targetVocabSize = targetSize;
        let ids = Array.from(this.textEncoder.encode(text, 'utf-8'));
        let currentVocabSize = 256;

        while (currentVocabSize < this.targetVocabSize) {
            const pairCounts = new Map();
            const singleCounts = new Map();
            let totalPairs = 0;

            // 1. Считаем поштучные частоты токенов
            for (let id of ids) {
                singleCounts.set(id, (singleCounts.get(id) || 0) + 1);
            }

            // 2. Считаем частоты смежных пар
            for (let i = 0; i < ids.length - 1; i++) {
                const pair = `${ids[i]},${ids[i+1]}`;
                pairCounts.set(pair, (pairCounts.get(pair) || 0) + 1);
                totalPairs++;
            }

            if (totalPairs === 0) break;

            let bestPair = null;
            let maxScore = -1.0;

            // 3. Вычисляем энтропийный скор для каждой пары
            for (let [pair, count] of pairCounts) {
                if (count < 2) continue; // Игнорируем случайные единичные совпадения

                const [id1, id2] = pair.split(',').map(Number);
                const count1 = singleCounts.get(id1);
                const count2 = singleCounts.get(id2);

                // Коэффициент Жаккара для вероятностей + логарифмическое взвешивание частоты
                // Это защищает алгоритм от склеивания слишком редких "мусорных" сочетаний
                const jaccardInertia = count / (count1 + count2 - count);
                const entropyWeight = Math.log2(count);
                const score = jaccardInertia * entropyWeight;

                if (score > maxScore) {
                    maxScore = score;
                    bestPair = pair;
                }
            }

            // Если не нашли подходящих кандидатов для слияния, выходим
            if (!bestPair) break;

            // 4. Регистрируем новый токен в системе
            const [id1, id2] = bestPair.split(',').map(Number);
            const newId = currentVocabSize;
            this.merges.set(bestPair, newId);

            const bytes1 = this.inverseVocab.get(id1);
            const bytes2 = this.inverseVocab.get(id2);
            const combinedBytes = new Array(...bytes1, ...bytes2);

            this.vocab.set(combinedBytes.toString(), newId);
            this.inverseVocab.set(newId, combinedBytes);

            // 5. Выполняем физическое слияние пары в текущем обучающем корпусе
            let newIds = new Array();
            let i = 0;
            while (i < ids.length) {
                if (i < ids.length - 1 && ids[i] === id1 && ids[i+1] === id2) {
                    newIds.push(newId);
                    i += 2;
                } else {
                    newIds.push(ids[i]);
                    i += 1;
                }
            }
            ids = newIds;
            currentVocabSize++;
        }
        return (`Энтропийное обучение завершено. Итоговый словарь: ${currentVocabSize}`);
    }

    // Кодирование входящего текста в Uint32Array для GPU
    encode(text) {
        let ids = Array.from(this.textEncoder.encode(text, 'utf-8'));
        let changed = true;

        while (changed) {
            changed = false;
            let newIds = new Array();
            let i = 0;

            while (i < ids.length) {
                if (i < ids.length - 1) {
                    const pair = `${ids[i]},${ids[i+1]}`;
                    if (this.merges.has(pair)) {
                        newIds.push(this.merges.get(pair));
                        i += 2;
                        changed = true;
                        continue;
                    }
                }
                newIds.push(ids[i]);
                i += 1;
            }
            ids = newIds;
        }
        return new Uint32Array(ids);
    }

    // Декодирование u32-блоков с GPU обратно в человеческий текст
    decode(idsArray) {
        let resultBytes = new Array();
        for (let i = 0; i < idsArray.length; i++) {
            const id = idsArray[i];
            const bytes = this.inverseVocab.get(id);
            if (bytes) {
                resultBytes.push(...bytes);
            }
        }
        return this.textDecoder.decode(new Uint8Array(resultBytes));
    }
}


In [ ]:
text = "🚀 Квантовая механика — это раздел фундаментальной физики, описывающий законы движения и взаимодействия материи на микроскопическом уровне (атомы, элементарные частицы). В отличие от классической физики, она оперирует вероятностями, квантовыми скачками и утверждает, что микрообъекты обладают свойствами как волн, так и частиц.4 фундаментальных принципаКорпускулярно-волновой дуализм: Любой объект проявляет себя и как волна, и как частица. Например, свет состоит из «порций» — фотонов, но распространяется по законам волновой интерференции.Принцип неопределенности Гейзенберга: Невозможно одновременно с абсолютной точностью узнать точное положение частицы и её скорость (или импульс).Принцип суперпозиции: Квантовая система может находиться сразу во всех возможных состояниях, пока мы её не измерим. Яркий пример — мысленный эксперимент с котом Шрёдингера.Вероятностный характер: Квантовая механика предсказывает не точный результат события, а вероятность того или иного исхода, которые описываются волновой функцией (уравнение Шрёдингера).Ключевые терминыКвант: Минимальная неделимая порция энергии или физической величины.Спин: Собственный внутренний момент импульса элементарной частицы, не связанный с её движением в пространстве.Квантовая запутанность: Состояние, при котором две частицы оказываются неразрывно связанными. Измерение состояния одной мгновенно определяет состояние другой, даже если они находятся на огромном расстоянии.Где это применяется?Понимание квантовых законов лежит в основе многих современных технологий:Полупроводники и электроника: Все микрочипы и процессоры в компьютерах созданы с использованием квантовой физики.Лазеры: Работают на основе вынужденного излучения фотонов.Квантовые компьютеры: Вычислительные устройства, использующие кубиты (квантовые биты, находящиеся в суперпозиции), что позволяет решать сверхсложные задачи.Чтобы детальнее изучить теорию и математический аппарат, обратитесь к академическому курсу Квантовая механика Teach-in или популярным статьям Ядерная физика в интернете."
>2**16
tokenizer = new BinNetEntropyTokenizer();
>tokens = tokenizer.encode(text)
>tokens.length
>text = tokenizer.decode(tokens)
>tokenizer.train(text, 2**16)
>tokenizer.train(text, 2**16)
>tokenizer.train(text, 2**16)
>tokens = tokenizer.encode(text)
>tokens.length
>text = tokenizer.decode(tokens);
// >tokenizer.vocab

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>2**16</label>

65536

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>tokens = tokenizer.encode(text)</label>

240,159,154,128,32,208,154,208,178,208,176,208,189,209,130,208,190,208,178,208,176,209,143,32,208,188,208,181,209,133,208,176,208,189,208,184,208,186,208,176,32,226,128,148,32,209,141,209,130,208,190,32,209,128,208,176,208,183,208,180,208,181,208,187,32,209,132,209,131,208,189,208,180,208,176,208,188,208,181,208,189,209,130,208,176,208,187,209,140,208,189,208,190,208,185,32,209,132,208,184,208,183,208,184,208,186,208,184,44,32,208,190,208,191,208,184,209,129,209,139,208,178,208,176,209,142,209,137,208,184,208,185,32,208,183,208,176,208,186,208,190,208,189,209,139,32,208,180,208,178,208,184,208,182,208,181,208,189,208,184,209,143,32,208,184,32,208,178,208,183,208,176,208,184,208,188,208,190,208,180,208,181,208,185,209,129,209,130,208,178,208,184,209,143,32,208,188,208,176,209,130,208,181,209,128,208,184,208,184,32,208,189,208,176,32,208,188,208,184,208,186,209,128,208,190,209,129,208,186,208,190,208,191,208,184,209,135,208,181,209,129,208,186,208,190,208,188,32,209,131,209,128,208,190,2

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>tokens.length</label>

...

240,159,154,128,267,332,357,282,514,297,263,273,331,262,286,265,548,577,357,32,573,290,431,574,390,306,32,276,262,290,262,286,262,334,261,288,505,265,362,262,306,370,546,268,351,262,321,487,267,184,284,290,265,262,287,261,430,306,346,388,297,385,292,394,267,189,265,297,262,286,451,333,261,288,420,261,287,32,512,261,282,495,549,385,261,287,268,334,406,562,530,508,329,365,426,268,41,325,146,267,190,256,434,380,267,190,256,295,571,515,420,261,306,32,276,262,290,262,286,262,334,546,265,267,190,288,292,262,541,359,477,390,280,269,384,334,286,282,331,357,282,268,384,482,265,271,286,265,384,267,184,32,531,282,292,321,428,359,467,297,262,286,451,261,313,293,263,457,268,267,190,313,571,428,361,348,261,306,346,265,384,371,284,261,285,264,334,256,369,267,184,32,426,46,52,574,381,335,391,274,262,288,265,208,154,398,441,333,532,390,45,282,261,285,390,282,261,306,303,270,265,285,262,290,287,298,155,275,313,261,306,267,190,313,293,263,457,296,451,269,282,480,359,324,263,313,269,267,184,371,284,261,28

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>tokens.length</label>

1222

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>text = tokenizer.decode(tokens)</label>

🚀 Квантовая механика — это раздел фундаментальной физики, описывающий законы движения и взаимодействия материи на микроскопическом уровне (атомы, элементарные частицы). В отличие от классической физики, она оперирует вероятностями, квантовыми скачками и утверждает, что микрообъекты обладают свойствами как волн, так и частиц.4 фундаментальных принципаКорпускулярно-волновой дуализм: Любой объект проявляет себя и как волна, и как частица. Например, свет состоит из «порций» — фотонов, но распространяется по законам волновой интерференции.Принцип неопределенности Гейзенберга: Невозможно одновременно с абсолютной точностью узнать точное положение частицы и её скорость (или импульс).Принцип суперпозиции: Квантовая система может находиться сразу во всех возможных состояниях, пока мы её не измерим. Яркий пример — мысленный эксперимент с котом Шрёдингера.Вероятностный характер: Квантовая механика предсказывает не точный результат события, а вероятность того или иного исхода, которые описываются 

In [ ]:
BinTokenizer = class BinTokenizer extends BinNet  { 
    constructor(config = {}) {
        super(...arguments);
        BinNet.WebGPU.clear();
              // Окно: 32 байта = 256 бит
        this.blocksPerPatch = config.blocksPerPatch || 8;   // 256 бит / 32 бита = 8 чисел Uint32 на признаки
        this.threshold = config.threshold || 1.2;
        this.nGramOrder = config.nGramOrder || 3;
        this.embeddingSize = config.embeddingSize || 32; // количество 32 битных блоков
        this.deep = config.deep || 2; // Количество слоев энкодера / декодера
        this.maxBytes = this.blocksPerPatch * 4;  // количество Uint8 байт в патче

        this.freqTable = {};
        this.encoder = new TextEncoder();
        this.decoder = new TextDecoder();
        let start = this.blocksPerPatch + 1;
        let step = (this.embeddingSize - start) / this.deep;
        this.encoder_layers = [];
        this.decoder_layers = [];
        for(let l = 0; l < this.deep; l++){
            let next = Math.min(Math.ceil(start + step), this.embeddingSize);
            this.encoder_layers.push(new BinLayer({in_size: start, out_size: next, id: l}));
            this.decoder_layers.unshift(new BinLayer({in_size: next, out_size: start, id: this.deep * 2 - l - 1}));
            start = next;
        }
        this.masks = {};
    }
    get layers(){
        return [...this.encoder_layers, ...this.decoder_layers];
    }
    _getUtf8CharLength(firstByte) {
        if (firstByte < 0x80) return 1;
        if (firstByte >= 0x80 && firstByte < 0xC0)
            throw new Error(`Не допустимый код для первого байта: ${firstByte.toString(2).padStart(8,'0')}`);
        if (firstByte >= 0xC0 && firstByte < 0xE0) return 2;
        if (firstByte >= 0xE0 && firstByte < 0xF0) return 3;
        if (firstByte >= 0xF0) return 4;
        if (firstByte >= 0xF8)
            throw new Error(`Не допустимый код в UTF-8: ${firstByte.toString(2).padStart(8,'0')}`);
        return 1;
    }
    trainPatcher(text) {
        const bytes = this.encoder.encode(text);
        for (let i = 0; i < bytes.length - this.nGramOrder; i++) {
            const context = bytes.slice(i, i + this.nGramOrder - 1).join(',');
            const nextByte = bytes[i + this.nGramOrder - 1];
            let ctxData = this.freqTable[context];
            if(!ctxData)
                ctxData = this.freqTable[context] = {}
            ctxData[nextByte] = (ctxData[nextByte] || 0) + 1;
        }
    }
    _calculateEntropy(contextBytes) {
        const contextKey = contextBytes.slice(-(this.nGramOrder - 1)).join(',');
        const ctxData = this.freqTable[contextKey];
        if (!ctxData) return 3.0;
        let entropy = 0;
        let total = Object.values(ctxData).sum();
        for (const byte in ctxData) {
            const probability = ctxData[byte] / total;
            entropy -= probability * Math.log2(probability);
        }
        return entropy;
    }
    _textToPatches(text) {
        const bytes = this.encoder.encode(text);
        const patches = [];
        let currentPatch = [];
        const isSpaceByte = (b) => b === 32 || b === 10 || b === 9 || b === 13;
        for (let i = 0; i < bytes.length; i++) {
            currentPatch.push(bytes[i]);
            if (i < bytes.length - 1) {
                const currentByte = bytes[i];
                const nextByte = bytes[i + 1];
                const isNextContinuation = (nextByte >= 0x80 && nextByte <= 0xBF);
                const isCurrentLeading = (currentByte >= 0xC0);
                const isInsideUtf8Char = isCurrentLeading || isNextContinuation;
                const isSpaceBoundary = isSpaceByte(currentByte) && !isSpaceByte(nextByte);
                const isWordEnding = !isSpaceByte(currentByte) && isSpaceByte(nextByte);
                const entropy = this._calculateEntropy(currentPatch);
                const isEntropyHigh = entropy > this.threshold || isSpaceBoundary || isWordEnding;
                const isNextPushOverLimit = (currentPatch.length >= this.maxBytes);
                const nextSymbolLength = isNextContinuation ? 0 : this._getUtf8CharLength(nextByte);
                const willNextSymbolFit = (currentPatch.length + nextSymbolLength) <= this.maxBytes;

                if (((isEntropyHigh && willNextSymbolFit) || isNextPushOverLimit) && !isInsideUtf8Char) {
                    patches.push(new Uint8Array(currentPatch));
                    currentPatch = [];
                }
            }
        }
        if (currentPatch.length > 0) 
            patches.push(new Uint8Array(currentPatch));
        return patches;
    }
    _patchToToken(patch){
        let token = new Uint32Array(this.blocksPerPatch + 1);
        
        let start = 0, b = 0;
        while (start < patch.length) {
            let length = this._getUtf8CharLength(patch[start]);
            let block = patch.subarray(start, start + length);
            for (let i = 0; i < 4; i++) {
                token[b] = (token[b] << 8) | block[i];
            }
            start += length;
            b++;
        }
        
        // for(let p = 0, b = 0; p < patch.length; p += 4, b++){
        //     let block = patch.subarray(p, p + 4);
        //     if(!block.length)
        //         break;
        //     for (let i = 0; i < 4; i++) {
        //         token[b] = (token[b] << 8) | block[i];
        //     }                
        // }
        // let mask = this._getMaskBlock(patch.length);

        let mask = this._getMaskBlock(b);
        token[this.blocksPerPatch] = mask;
        return token;
    }
    _getMaskBlock(patch_size){
        let mask = this.masks[patch_size];
        if(!mask)
            this.masks[patch_size] = mask = parseInt('1'.repeat(patch_size).padEnd(32, '0'), 2);
        return mask
    }
    // ============================================================================
    // ЭТАП 2: УПАКОВКА В UINT32 (И ПРИЗНАКИ, И МАСКА)
    // ============================================================================
    tokenize(text) {
        const dynamicPatches = this._textToPatches(text);
        const numPatches = dynamicPatches.length;
        
        // По 8 чисел Uint32 на признаки каждого патча
        const uint32Tensor = new Uint32Array(numPatches * this.blocksPerPatch);
        
        // Ровно 1 число Uint32 на маску внимания каждого патча (32 бита кодируют 32 байта)
        const uint32Mask = new Uint32Array(numPatches);

        for (let pIdx = 0; pIdx < numPatches; pIdx++) {
            const patch = dynamicPatches[pIdx];
            
            // --- 1. Побитовая сборка маски внимания ---
            let maskWord = 0>>>0;
            for (let bIdx = 0; bIdx < this.maxBytes; bIdx++) {
                const isValidByte = bIdx < patch.length ? 1 : 0;
                // Сдвигаем влево и выставляем бит (от старшего к младшему)
                maskWord = (maskWord << 1) | isValidByte;
            }
            uint32Mask[pIdx] = maskWord >>> 0;

            // --- 2. Побитовая сборка признаков (8 слов Uint32) ---
            const tensorOffset = pIdx * this.blocksPerPatch;
            for (let wIdx = 0; wIdx < this.blocksPerPatch; wIdx++) {
                let currentWord = 0 >>> 0;
                
                for (let byteOffset = 0; byteOffset < 4; byteOffset++) {
                    const globalByteIdx = (wIdx * 4) + byteOffset;
                    const byte = globalByteIdx < patch.length ? patch[globalByteIdx] : 0x00;
                    currentWord = (currentWord << 8) | byte;
                }
                uint32Tensor[tensorOffset + wIdx] = currentWord >>> 0;
            }
        }

        return {dynamicPatches, uint32Tensor, uint32Mask, numPatches};
    }
    decodeToken(input){ // uint32TokenMaskVector  9 блоков по 32 бита
        let uint32Tensor = input.subarray(0, this.blocksPerPatch);
        let uint32Mask = input.subarray(this.blocksPerPatch, 1)
        return this.decodeWithMask(uint32Tensor, uint32Mask, 1)
    }
    encode(input){ // uint32TokenMaskVector
        return this.encoder_layers.reduce((res, layer, i) =>layer.forward(res), input);
    }
    decode(input){ // uint32EmbVector
        return this.decoder_layers.reduce((res, layer, i) =>layer.forward(res), input);
    }
    forward(input){ // uint32TokenMaskVector
        return this.decode(this.encode(input));
    }
    get error(){
        let errors = [
            this.decoder_layers.last.error
        ].flat();
        return Promise.all(errors).then(res=>{
            return res.avg()
        })
    }
    train(input){
        this.forward(input);
        let target = this.decoder_layers.toReversed().reduce((res, layer)=>layer.back(res), input);
        this.encoder_layers.toReversed().reduce((res, layer)=>layer.back(res), target);       
    }

  
    // ============================================================================
    // ЭТАП 3: ДЕКОДИРОВАНИЕ С ПОБИТОВОЙ UINT32 МАСКИ В ИСХОДНЫЙ ТЕКСТ
    // ============================================================================

    /** Восстановление текста по 256-битному тензору и 32-битной маске */
    decodeWithMask(uint32Tensor, uint32Mask, numPatches) {
        const finalBytes = [];

        for (let pIdx = 0; pIdx < numPatches; pIdx++) {
            const tensorOffset = pIdx * this.blocksPerPatch;
            const maskWord = uint32Mask[pIdx];

            for (let bIdx = 0; bIdx < this.maxBytes; bIdx++) {
                // Проверяем бит маски внимания: сдвигаем маску и смотрим на младший бит
                const isByteValid = (maskWord >>> (31 - bIdx)) & 1;
                if (isByteValid === 0) 
                    continue; // Если бит равен 0 — это паддинг, пропускаем

                const wordIdx = Math.floor(bIdx / 4);
                const byteInWordIdx = bIdx % 4;
                const currentWord = uint32Tensor[tensorOffset + wordIdx];

                const shift = (3 - byteInWordIdx) * 8;
                const byte = (currentWord >>> shift) & 0xFF;
                
                finalBytes.push(byte);
            }
        }

        return this.decoder.decode(new Uint8Array(finalBytes));
    }
}


// ПРОВЕРКА РЕЗУЛЬТАТОВ 

tokenizer = new BinTokenizer({ threshold: 1.618, embeddingSize: 32, deep: 2, blocksPerPatch: 7});

>tokenizer.encoder_layers.map(l=>({"in": l.in_size, "out": l.out_size}))

text = "🚀 Квантовая механика — это раздел фундаментальной физики, описывающий законы движения и взаимодействия материи на микроскопическом уровне (атомы, элементарные частицы). В отличие от классической физики, она оперирует вероятностями, квантовыми скачками и утверждает, что микрообъекты обладают свойствами как волн, так и частиц.4 фундаментальных принципаКорпускулярно-волновой дуализм: Любой объект проявляет себя и как волна, и как частица. Например, свет состоит из «порций» — фотонов, но распространяется по законам волновой интерференции.Принцип неопределенности Гейзенберга: Невозможно одновременно с абсолютной точностью узнать точное положение частицы и её скорость (или импульс).Принцип суперпозиции: Квантовая система может находиться сразу во всех возможных состояниях, пока мы её не измерим. Яркий пример — мысленный эксперимент с котом Шрёдингера.Вероятностный характер: Квантовая механика предсказывает не точный результат события, а вероятность того или иного исхода, которые описываются волновой функцией (уравнение Шрёдингера).Ключевые терминыКвант: Минимальная неделимая порция энергии или физической величины.Спин: Собственный внутренний момент импульса элементарной частицы, не связанный с её движением в пространстве.Квантовая запутанность: Состояние, при котором две частицы оказываются неразрывно связанными. Измерение состояния одной мгновенно определяет состояние другой, даже если они находятся на огромном расстоянии.Где это применяется?Понимание квантовых законов лежит в основе многих современных технологий:Полупроводники и электроника: Все микрочипы и процессоры в компьютерах созданы с использованием квантовой физики.Лазеры: Работают на основе вынужденного излучения фотонов.Квантовые компьютеры: Вычислительные устройства, использующие кубиты (квантовые биты, находящиеся в суперпозиции), что позволяет решать сверхсложные задачи.Чтобы детальнее изучить теорию и математический аппарат, обратитесь к академическому курсу Квантовая механика Teach-in или популярным статьям Ядерная физика в интернете."

tokenizer.trainPatcher(text);

tokens = tokenizer.tokenize(text);


<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>tokenizer.encoder_layers.map(l=>({"in": l.in_size, "out": l.out_size}))</label>

[
  {
    "in": 8,
    "out": 20
  },
  {
    "in": 20,
    "out": 32
  }
]

In [ ]:
UTF8_char_code_generator = function() {
    let char = [];
    let firstByte = Math.trunc(Math.random() * 0xF7 + 1);  // первый байт может быть в диапазоне 0 - 0xF7
    if (firstByte >= 0x80 && firstByte < 0xC0)  // коды зарезервированные для последующих байтов 
        firstByte &= 0x7F;    // превращаем в символ из одного байта
    char.push(firstByte);
    let size = (firstByte >= 0xC0) + (firstByte >= 0xE0) + (firstByte >= 0xF0);  //количество дополнительных кодов в символе
    for (let i = 0; i < size; i++) {
        let nextByte = Math.trunc(Math.random() * 0x3F + 1) | 0x80; // байт в формате последующих кодов символа
        char.push(nextByte);
    }
    // >tokenizer.decoder.decode(new Uint8Array(char))
    return char;
}
gen_patch = function(){
    let count_symbols = Math.trunc((Math.random() * tokenizer.blocksPerPatch)) || 1;
    let patch = [];
    for (let s = 0; s < count_symbols; s++) {
        patch.push(UTF8_char_code_generator());
    }
    return new Uint8Array(patch.flat());
}

In [ ]:
// ww = await Promise.all(tokenizer.layers.map(l=>l.bin_weights).flat())
// >ww
let learn_count = 10
let percent = 0;
let dim = (tokenizer.blocksPerPatch + 1) * 32;



percent = 0;
for(let i = 0; i < learn_count; i++){

    token = tokenizer._patchToToken(gen_patch());
//     >patch
// >BinNet.vec2bits(token);
    await tokenizer.train(token);
    let n_percent = Math.round(i / learn_count * 100);
    if(percent === n_percent)
        continue;
    await this.setProgress(percent = n_percent);
}

>error = await tokenizer.error;

// >ww = await Promise.all(tokenizer.layers.map(l=>l.bin_weights).flat())
// >ww = await Promise.all(tokenizer.layers.map(l=>l.bin_back_weights).flat())

percent = 0;
for(let i = 0; i < 10; i++){

    token = tokenizer._patchToToken(gen_patch());
> BinNet.vec2bits(token).replace(/\d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)
    result = tokenizer.forward(token);
    result = await BinNet.WebGPU.readData(result);
    res = BinNet.vec2bits(result).replace(/\d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)
    >res
    >error = BinNet.bitSimilarityUint32(token, result);
    
    let n_percent = Math.round(i / learn_count * 100);
    if(percent === n_percent)
        continue;
    await this.setProgress(percent = n_percent);
}



// >ww

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>error = await tokenizer.error</label>

0.10546875

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'> BinNet.vec2bits(token).replace(/d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)</label>

00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  10000000:00000000:00000000:00000000

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>res</label>

...

00110100:00000000:00000000:00000000  00111000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  11000000:00000000:00000000:00000000

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>res</label>

01111111:00000000:00000000:00000000  01101111:00000000:00000000:00000000  01101111:10000100:00000000:00000000  00100001:00100000:00000000:00000010  00000000:00000001:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  11100000:00000000:00000000:00000000

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>error = BinNet.bitSimilarityUint32(token, result)</label>

0.08984375

In [ ]:
tokenizer = new BinTokenizer({ threshold: 1.618, embeddingSize: 32, deep: 2, blocksPerPatch: 7});
// >tokenizer.encoder_layers.map(l=>({"in": l.in_size, "out": l.out_size}))
// >tokenizer.decoder_layers.map(l=>({"in": l.in_size, "out": l.out_size}))

example_count = 2
token = []
for ( let i = 0; i < example_count; i++) {
    token[i] = tokenizer._patchToToken(gen_patch());
    > BinNet.vec2bits(token[i]).replace(/\d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)
}
>tokenizer.layers.map(l=>l.weights.length)

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'> BinNet.vec2bits(token[i]).replace(/d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)</label>

00010110:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  10000000:00000000:00000000:00000000

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'> BinNet.vec2bits(token[i]).replace(/d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)</label>

01110000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  10000000:00000000:00000000:00000000

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>tokenizer.layers.map(l=>l.weights.length)</label>

[
  5120,
  20480,
  20480,
  5120
]

In [ ]:
let learn_count = 1
percent = 0;
for(let i = 0; i < example_count; i++){
    > BinNet.vec2bits(token[i]).replace(/\d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)
}
for(let i = 0; i < learn_count; i++){
    for(let e = 0; e < example_count; e++){
        await tokenizer.train(token[e]);
        let n_percent = Math.round(i*e / learn_count * 100);
        if(percent === n_percent)
            continue;
        await this.setProgress(percent = n_percent);
    }
}
for(let i = 0; i < example_count; i++){
    > BinNet.vec2bits(token[i]).replace(/\d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)
}
>error = await tokenizer.error;
percent = 0;
for(let i = 0; i < example_count; i++){
    > BinNet.vec2bits(token[i]).replace(/\d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)
    result = tokenizer.forward(token[i]);
    result = await BinNet.WebGPU.readData(result);
    res = BinNet.vec2bits(result).replace(/\d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)
    >res
    >error = BinNet.bitSimilarityUint32(token[i], result);
    
    let n_percent = Math.round(i / learn_count * 100);
    if(percent === n_percent)
        continue;
    await this.setProgress(percent = n_percent);
}


<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'> BinNet.vec2bits(token[i]).replace(/d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)</label>

00010110:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  10000000:00000000:00000000:00000000

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'> BinNet.vec2bits(token[i]).replace(/d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)</label>

01110000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  10000000:00000000:00000000:00000000

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'> BinNet.vec2bits(token[i]).replace(/d{8}/g,'$&:').replaceAll(': ','  ').slice(0, -1)</label>

...

01110000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  00000000:00000000:00000000:00000000  10000000:00000000:00000000:00000000

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>res</label>

11111111:11111111:11111111:11110111  11111111:11111111:10111111:11101011  11111111:11111101:10111111:11111101  11111111:11111111:11111111:11111111  00110100:10101001:10101111:11111101  11111011:10001111:10011011:10101010  10000101:00011111:10101110:10101011  11001011:01111000:00110100:11011000

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>error = BinNet.bitSimilarityUint32(token[i], result)</label>

0.75

In [ ]:
XLLM = class XLLM extends BinNet {
    static max = 2 ** 32 - 1;
    forward_data = {};
    update_data = {}
    constructor(config = {}) {
        super();
        this.deep = config.deep || 1;
        this.max_size = config.max_size || 2048;
        if(!Number.isInteger(this.deep) || this.deep<1)
            throw new Error('Количество слоев (deep) должно быть натуральным числом');
        BinNet.WebGPU.clear();
        this.size = Math.min(2 ** Math.max((this.deep - 2), 0), this.max_size);
        this.dim = this.size * 32;
        this.token_queue = [];
        this.token_output = [];
        this.layers = [];          // Слои
        let size = 1;
        while(size <= this.size){
            this.layers.push(new BinLayer({in_size: this.size, out_size: this.size, divider: size, id: this.layers.length}));
            size *= 2;
        }
        while(this.layers.length < this.deep){
            this.layers.push(new BinLayer({in_size: this.size, out_size: this.size, divider: 1, id: this.layers.length}));
        }
        this.input = BinNet.create_zeros_vector(this.size);
        this.target = BinNet.create_zeros_vector(this.size);
        BinNet.WebGPU.writeData(this.input, true, 'input');
        BinNet.WebGPU.writeData(this.target, true, 'target');
    }
    get param_count(){
        return this.layers.reduce((sum, layer) => sum + layer.param_count, 0);
    }
    async read(text = ''){
        let tokens = tokenizer.tokenize(text);
        // this.token_queue.push(...text.match(BinNet.gpt4RegexJS) || [])
        // this.token_queue.push(0);
        // >this.token_queue
    }
    async run(p = .1){
        if(!this.isRun){
            this.isRun = true;
            this.input.set(BinNet.create_random_vector(this.size));
            this.counter = 0;
        }
        let token = this.token_queue.shift();
        let result;
        if(token){
            // token = this._get_token_vector(token);
            this.input.set(token);
            let target = this.token_queue[0];
            if(target){
                this.target.set(target);
                result = this.train(this.input, this.target);
            }
        }
        if(!result)
            result = this.forward(this.input);   
        this.counter++;
        
        result = await BinNet.WebGPU.readData(result);
// >result    
        token = result.subarray(0, 1024/*this.emb_size*/);
        
        let patch = tokenizer.decode(token);

        this.token_output.push(patch);
        this.dispatchEvent(new CustomEvent('progress', { bubbles: true,  detail: { value: this.counter }, composed: true}))
        this.input.set(result);
        queueMicrotask(() => {
            if(this.isRun)
                this.run()
        });
    }
    stop(){
        this.isRun = false;
        // this.input = null;
    }
    forward(input) {
        return this.layers.reduce((res, layer, i) =>layer.forward(res), input);
    }

    // Обучение сети 
    train(input, target) {
        this.forward(input);
        this.layers.toReversed().reduce((res, layer)=>layer.back(res), target);
    }
    get error(){
        let errors = [
            this.layers.map(l=>l.error),
        ].flat();
        return Promise.all(errors).then(res=>{
            return res.avg()
        })
    }    
}

deepNet = new XLLM({deep: 30, max_size: 1024});

input = BinNet.create_random_vector(deepNet.size);
// >input
target = BinNet.create_random_vector(deepNet.size);
// >target
>deepNet.dim
>deepNet.layers.map(l=>l.weights.length)
>deepNet.param_count.toLocaleString()
>deepNet.dim

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>deepNet.dim</label>

32768

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>deepNet.layers.map(l=>l.weights.length)</label>

[
  33554432,
  16777216,
  8388608,
  4194304,
  2097152,
  1048576,
  524288,
  262144,
  131072,
  65536,
  32768,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432,
  33554432
]

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>deepNet.param_count.toLocaleString()</label>

22 547 529 728

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>deepNet.dim</label>

32768

In [ ]:
learn_count = 60
for (let epoch = 0; epoch < learn_count; epoch++) {
    // deepNet.forward(input)
      deepNet.train(input, target);
    //  result = deepNet.forward(input);
    // deepNet.forward(input);
    await this.setProgress(Math.round(epoch / learn_count * 100));
}

>"<h2>Выход сети ПОСЛЕ обучения:</h2>"
result = deepNet.forward(input);
>error = await deepNet.error;
// result = await BinNet.WebGPU.readData(result)
// >error = BinNet.bitSimilarityUint32(result, target);
// ww = await Promise.all(deepNet.layers.map(l=>l.bin_weights).flat())
// >ww

<div style='cursor: pointer' onclick='_findCodeEntry(this)'><h2>Выход сети ПОСЛЕ обучения:</h2></div>

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>error = await deepNet.error</label>

0.000018310546875

In [ ]:
deepNet.read('Мама мыла раму, а потом тщательно ее сушила до 18.30')

text = "Квантовая механика — это раздел фундаментальной физики, описывающий законы движения и взаимодействия материи на микроскопическом уровне (атомы, элементарные частицы). В отличие от классической физики, она оперирует вероятностями, квантовыми скачками и утверждает, что микрообъекты обладают свойствами как волн, так и частиц.4 фундаментальных принципаКорпускулярно-волновой дуализм: Любой объект проявляет себя и как волна, и как частица. Например, свет состоит из «порций» — фотонов, но распространяется по законам волновой интерференции.Принцип неопределенности Гейзенберга: Невозможно одновременно с абсолютной точностью узнать точное положение частицы и её скорость (или импульс).Принцип суперпозиции: Квантовая система может находиться сразу во всех возможных состояниях, пока мы её не измерим. Яркий пример — мысленный эксперимент с котом Шрёдингера.Вероятностный характер: Квантовая механика предсказывает не точный результат события, а вероятность того или иного исхода, которые описываются волновой функцией (уравнение Шрёдингера).Ключевые терминыКвант: Минимальная неделимая порция энергии или физической величины.Спин: Собственный внутренний момент импульса элементарной частицы, не связанный с её движением в пространстве.Квантовая запутанность: Состояние, при котором две частицы оказываются неразрывно связанными. Измерение состояния одной мгновенно определяет состояние другой, даже если они находятся на огромном расстоянии.Где это применяется?Понимание квантовых законов лежит в основе многих современных технологий:Полупроводники и электроника: Все микрочипы и процессоры в компьютерах созданы с использованием квантовой физики.Лазеры: Работают на основе вынужденного излучения фотонов.Квантовые компьютеры: Вычислительные устройства, использующие кубиты (квантовые биты, находящиеся в суперпозиции), что позволяет решать сверхсложные задачи.Чтобы детальнее изучить теорию и математический аппарат, обратитесь к академическому курсу Квантовая механика Teach-in или популярным статьям Ядерная физика в интернете."
deepNet.read(text)

In [ ]:
torus.manual_seed(0);
torus.manual_seed(22);
class DeepTargetProp1BitNetwork {
    /**
     * @param {Array<number>} layerDims - Массив размеров слоев, например [4, 8, 8, 4, 2]
     * где layerDims[0] - вход, layerDims[4] - выход сети.
     */
    constructor(layerDims) {
        this.layerDims = layerDims;
        this.numLayers = layerDims.length - 1; // 4 слоя весов

        // Инициализируем массивы для хранения весов и теневых счетчиков
        this.W = [];          // Прямые веса
        this.V = [];          // Обратные веса (V[1] проецирует назад на уровень 1)

        // Массивы для хранения активаций во время прямого хода
        this.activations = Array(this.layerDims.length).fill(null);

        for (let i = 0; i < this.numLayers; i++) {
            const inDim = layerDims[i];
            const outDim = layerDims[i + 1];

            // Создаем прямой слой i
            this.W.push(Array.from({ length: outDim }, () => new Int8Array(inDim).fill(0)));

            // Создаем обратный слой для проецирования назад (из outDim в inDim)
            // Обратные слои нужны только для скрытых слоев (для слоев 1, 2, 3)
            if (i > 0) {
                this.V.push(Array.from({ length: inDim }, () => new Int8Array(outDim).fill(0)));
            } else {
                this.V.push(null); // Для самого первого слоя обратный слой не нужен
            }
        }
    }

    // Базовый матричный проход для одного 1-битного слоя
    _layerForward(inputs, weights, step) {
step = "<h3>FORWARD step:"+ step +"</h3>"
>step
        let outputs = weights.map(row => {
            let sum = 0;
            for (let i = 0; i < inputs.length; i++) 
                sum += inputs[i] * row[i]; // Логика XNOR
            return sign(sum);
        });
        
        >inputs.join('');
        weights = printW(weights);
        >weights
        >outputs.join('');
        return Int8Array.from(outputs);
    }

    // Прямой ход через все 4 слоя сети
    forward(x) {
        this.activations[0] = Int8Array.from(x); // Входные данные (Уровень 0)
        
        for (let i = 0; i < this.numLayers; i++) {
            this.activations[i + 1] = this._layerForward(this.activations[i], this.W[i], i);
        }
        
        return this.activations[this.numLayers]; // Возвращаем финальный выход (Уровень 4)
    }

    // Обучение сети методом последовательного каскада мишеней
    train(x, targetOutput) {
        // 1. Прямой проход для генерации текущих активаций на всех слоях
        this.forward(x);

        // Массив для хранения мишеней (желаемых состояний) для каждого уровня активаций
        const targets = Array(this.layerDims.length).fill(null);
        targets[this.numLayers] = Int8Array.from(targetOutput); // Финальная мишень известна из разметки
>"<h1>РАЗВОРОТ</h1>"
        // 2. ОБРАТНЫЙ ХОД: Генерация мишеней для скрытых слоев и обучение обратных слоев V
        for (let i = this.numLayers - 1; i >= 1; i--) {
            const currentH = this.activations[i];     // Текущая реальная активация скрытого слоя
            const nextY = this.activations[i + 1];     // Текущий реальный выход следующего слоя
            const nextTarget = targets[i + 1];         // Желаемый выход следующего слоя (мишень)

            // А. Шаг обучения для обратного слоя V[i].
            // Он учится восстанавливать реальный currentH из реального nextY.
            this._updateLayerWeights(nextY, currentH, this.V[i], -i);

            // Б. Генерация мишени для текущего слоя i.
            // Пропускаем идеальную мишень следующего слоя (nextTarget) назад через обученный V[i]
            targets[i] = this._layerForward(nextTarget, this.V[i], -i);
        }

        // 3. ЛОКАЛЬНОЕ ОБНОВЛЕНИЕ ХЭББА: Каждый из 4-х прямых слоев обучается независимо
        for (let i = 0; i < this.numLayers; i++) {
            const layerInput = this.activations[i];  // Реальный вход в слой i
            const layerTarget = targets[i + 1];      // Сгенерированная или финальная мишень для слоя i

            this._updateLayerWeights(layerInput, layerTarget, this.W[i], i);
        }
    }

    // Универсальное правило Хэбба с ограничением накопления на базе XNOR
    _updateLayerWeights(inputs, outputs, weights, step) {
>"<h2>Обновляем веса</h2>"
>step
>printW(weights)
        for (let i = 0; i < weights.length; i++) {
            let random = torus.generator();
            if(random >.01)
                for (let j = 0; j < inputs.length; j++) {
                    weights[i][j] = (inputs[j] === outputs[i])?1:0;
                }
        }
        >printW(weights)
    }
}

// === ТЕСТИРОВАНИЕ СЕТИ ИЗ 4 СЛОЕВ ===

// Задаем архитектуру: Вход (4) -> Скрытый 1 (8) -> Скрытый 2 (8) -> Скрытый 3 (4) -> Выход (2)

const deepNet = new DeepTargetProp1BitNetwork(networkDims);
inputPattern = new Uint32Array(Math.ceil(networkDims[0] / 32)).map(_=>torus.generator() * (2 ** networkDims[0]));
inputPattern = input.split('');
targetPattern = new Uint32Array(Math.ceil(networkDims.last / 32)).map(_=>torus.generator() * (2 ** networkDims.last));
targetPattern = target.split('')
>input
>target

// Обучаем сеть (100 итераций)
for (let epoch = 0; epoch < 1; epoch++) {
    deepNet.train(inputPattern, targetPattern);
}

>"Выход сети ПОСЛЕ обучения:"
>result = deepNet.forward(inputPattern).join('');
>target === result;



In [ ]:
deepNet.stop();
deepNet.removeEventListener('progress', handler);

// >Object.keys(deepNet.vocab)
// >Object.values(deepNet.vocab).map(v=>XLLM.vec2bits(v))


In [ ]:
torus.manual_seed(0);
torus.manual_seed(22);
class DeepTargetProp1BitNetwork {
    /**
     * @param {Array<number>} layerDims - Массив размеров слоев, например [4, 8, 8, 4, 2]
     * где layerDims[0] - вход, layerDims[4] - выход сети.
     */
    constructor(layerDims) {
        this.layerDims = layerDims;
        this.numLayers = layerDims.length - 1; // 4 слоя весов

        // Инициализируем массивы для хранения весов и теневых счетчиков
        this.W = [];          // Прямые веса
        this.V = [];          // Обратные веса (V[1] проецирует назад на уровень 1)

        // Массивы для хранения активаций во время прямого хода
        this.activations = Array(this.layerDims.length).fill(null);

        for (let i = 0; i < this.numLayers; i++) {
            const inDim = layerDims[i];
            const outDim = layerDims[i + 1];

            // Создаем прямой слой i
            this.W.push(Array.from({ length: outDim }, () => new Int8Array(inDim).fill(0)));

            // Создаем обратный слой для проецирования назад (из outDim в inDim)
            // Обратные слои нужны только для скрытых слоев (для слоев 1, 2, 3)
            if (i > 0) {
                this.V.push(Array.from({ length: inDim }, () => new Int8Array(outDim).fill(0)));
            } else {
                this.V.push(null); // Для самого первого слоя обратный слой не нужен
            }
        }
    }

    // Базовый матричный проход для одного 1-битного слоя
    _layerForward(inputs, weights, step) {
step = "<h3>FORWARD step:"+ step +"</h3>"
>step
        let outputs = weights.map(row => {
            let sum = 0;
            for (let i = 0; i < inputs.length; i++) 
                sum += inputs[i] * row[i]; // Логика XNOR
            return sign(sum);
        });
        
        >inputs.join('');
        weights = printW(weights);
        >weights
        >outputs.join('');
        return Int8Array.from(outputs);
    }

    // Прямой ход через все 4 слоя сети
    forward(x) {
        this.activations[0] = Int8Array.from(x); // Входные данные (Уровень 0)
        
        for (let i = 0; i < this.numLayers; i++) {
            this.activations[i + 1] = this._layerForward(this.activations[i], this.W[i], i);
        }
        
        return this.activations[this.numLayers]; // Возвращаем финальный выход (Уровень 4)
    }

    // Обучение сети методом последовательного каскада мишеней
    train(x, targetOutput) {
        // 1. Прямой проход для генерации текущих активаций на всех слоях
        this.forward(x);

        // Массив для хранения мишеней (желаемых состояний) для каждого уровня активаций
        const targets = Array(this.layerDims.length).fill(null);
        targets[this.numLayers] = Int8Array.from(targetOutput); // Финальная мишень известна из разметки
>"<h1>РАЗВОРОТ</h1>"
        // 2. ОБРАТНЫЙ ХОД: Генерация мишеней для скрытых слоев и обучение обратных слоев V
        for (let i = this.numLayers - 1; i >= 1; i--) {
            const currentH = this.activations[i];     // Текущая реальная активация скрытого слоя
            const nextY = this.activations[i + 1];     // Текущий реальный выход следующего слоя
            const nextTarget = targets[i + 1];         // Желаемый выход следующего слоя (мишень)

            // А. Шаг обучения для обратного слоя V[i].
            // Он учится восстанавливать реальный currentH из реального nextY.
            this._updateLayerWeights(nextY, currentH, this.V[i], -i);

            // Б. Генерация мишени для текущего слоя i.
            // Пропускаем идеальную мишень следующего слоя (nextTarget) назад через обученный V[i]
            targets[i] = this._layerForward(nextTarget, this.V[i], -i);
        }

        // 3. ЛОКАЛЬНОЕ ОБНОВЛЕНИЕ ХЭББА: Каждый из 4-х прямых слоев обучается независимо
        for (let i = 0; i < this.numLayers; i++) {
            const layerInput = this.activations[i];  // Реальный вход в слой i
            const layerTarget = targets[i + 1];      // Сгенерированная или финальная мишень для слоя i

            this._updateLayerWeights(layerInput, layerTarget, this.W[i], i);
        }
    }

    // Универсальное правило Хэбба с ограничением накопления на базе XNOR
    _updateLayerWeights(inputs, outputs, weights, step) {
>"<h2>Обновляем веса</h2>"
>step
>printW(weights)
        for (let i = 0; i < weights.length; i++) {
            let random = torus.generator();
            if(random >.01)
                for (let j = 0; j < inputs.length; j++) {
                    weights[i][j] = (inputs[j] === outputs[i])?1:0;
                }
        }
        >printW(weights)
    }
}

// === ТЕСТИРОВАНИЕ СЕТИ ИЗ 4 СЛОЕВ ===

// Задаем архитектуру: Вход (4) -> Скрытый 1 (8) -> Скрытый 2 (8) -> Скрытый 3 (4) -> Выход (2)

const deepNet = new DeepTargetProp1BitNetwork(networkDims);
inputPattern = new Uint32Array(Math.ceil(networkDims[0] / 32)).map(_=>torus.generator() * (2 ** networkDims[0]));
inputPattern = input.split('');
targetPattern = new Uint32Array(Math.ceil(networkDims.last / 32)).map(_=>torus.generator() * (2 ** networkDims.last));
targetPattern = target.split('')
>input
>target

// Обучаем сеть (100 итераций)
for (let epoch = 0; epoch < 1; epoch++) {
    deepNet.train(inputPattern, targetPattern);
}

>"Выход сети ПОСЛЕ обучения:"
>result = deepNet.forward(inputPattern).join('');
>target === result;



In [ ]:
torus.manual_seed(0);
torus.manual_seed(22);
class DeepTargetProp1BitNetwork {
    /**
     * @param {Array<number>} layerDims - Массив размеров слоев, например [4, 8, 8, 4, 2]
     * где layerDims[0] - вход, layerDims[4] - выход сети.
     */
    constructor(layerDims) {
        this.layerDims = layerDims;
        this.numLayers = layerDims.length - 1; // 4 слоя весов

        // Инициализируем массивы для хранения весов и теневых счетчиков
        this.W = [];          // Прямые веса
        this.V = [];          // Обратные веса (V[1] проецирует назад на уровень 1)

        // Массивы для хранения активаций во время прямого хода
        this.activations = Array(this.layerDims.length).fill(null);

        for (let i = 0; i < this.numLayers; i++) {
            const inDim = layerDims[i];
            const outDim = layerDims[i + 1];

            // Создаем прямой слой i
            this.W.push(Array.from({ length: outDim }, () => new Int8Array(inDim).fill(0)));

            // Создаем обратный слой для проецирования назад (из outDim в inDim)
            // Обратные слои нужны только для скрытых слоев (для слоев 1, 2, 3)
            if (i > 0) {
                this.V.push(Array.from({ length: inDim }, () => new Int8Array(outDim).fill(0)));
            } else {
                this.V.push(null); // Для самого первого слоя обратный слой не нужен
            }
        }
    }

    // Базовый матричный проход для одного 1-битного слоя
    _layerForward(inputs, weights, step) {
step = "<h3>FORWARD step:"+ step +"</h3>"
>step
        let outputs = weights.map(row => {
            let sum = 0;
            for (let i = 0; i < inputs.length; i++) 
                sum += inputs[i] * row[i]; // Логика XNOR
            return sign(sum);
        });
        
        >inputs.join('');
        weights = printW(weights);
        >weights
        >outputs.join('');
        return Int8Array.from(outputs);
    }

    // Прямой ход через все 4 слоя сети
    forward(x) {
        this.activations[0] = Int8Array.from(x); // Входные данные (Уровень 0)
        
        for (let i = 0; i < this.numLayers; i++) {
            this.activations[i + 1] = this._layerForward(this.activations[i], this.W[i], i);
        }
        
        return this.activations[this.numLayers]; // Возвращаем финальный выход (Уровень 4)
    }

    // Обучение сети методом последовательного каскада мишеней
    train(x, targetOutput) {
        // 1. Прямой проход для генерации текущих активаций на всех слоях
        this.forward(x);

        // Массив для хранения мишеней (желаемых состояний) для каждого уровня активаций
        const targets = Array(this.layerDims.length).fill(null);
        targets[this.numLayers] = Int8Array.from(targetOutput); // Финальная мишень известна из разметки
>"<h1>РАЗВОРОТ</h1>"
        // 2. ОБРАТНЫЙ ХОД: Генерация мишеней для скрытых слоев и обучение обратных слоев V
        for (let i = this.numLayers - 1; i >= 1; i--) {
            const currentH = this.activations[i];     // Текущая реальная активация скрытого слоя
            const nextY = this.activations[i + 1];     // Текущий реальный выход следующего слоя
            const nextTarget = targets[i + 1];         // Желаемый выход следующего слоя (мишень)

            // А. Шаг обучения для обратного слоя V[i].
            // Он учится восстанавливать реальный currentH из реального nextY.
            this._updateLayerWeights(nextY, currentH, this.V[i], -i);

            // Б. Генерация мишени для текущего слоя i.
            // Пропускаем идеальную мишень следующего слоя (nextTarget) назад через обученный V[i]
            targets[i] = this._layerForward(nextTarget, this.V[i], -i);
        }

        // 3. ЛОКАЛЬНОЕ ОБНОВЛЕНИЕ ХЭББА: Каждый из 4-х прямых слоев обучается независимо
        for (let i = 0; i < this.numLayers; i++) {
            const layerInput = this.activations[i];  // Реальный вход в слой i
            const layerTarget = targets[i + 1];      // Сгенерированная или финальная мишень для слоя i

            this._updateLayerWeights(layerInput, layerTarget, this.W[i], i);
        }
    }

    // Универсальное правило Хэбба с ограничением накопления на базе XNOR
    _updateLayerWeights(inputs, outputs, weights, step) {
>"<h2>Обновляем веса</h2>"
>step
>printW(weights)
        for (let i = 0; i < weights.length; i++) {
            let random = torus.generator();
            if(random >.01)
                for (let j = 0; j < inputs.length; j++) {
                    weights[i][j] = (inputs[j] === outputs[i])?1:0;
                }
        }
        >printW(weights)
    }
}

// === ТЕСТИРОВАНИЕ СЕТИ ИЗ 4 СЛОЕВ ===

// Задаем архитектуру: Вход (4) -> Скрытый 1 (8) -> Скрытый 2 (8) -> Скрытый 3 (4) -> Выход (2)

const deepNet = new DeepTargetProp1BitNetwork(networkDims);
inputPattern = new Uint32Array(Math.ceil(networkDims[0] / 32)).map(_=>torus.generator() * (2 ** networkDims[0]));
inputPattern = input.split('');
targetPattern = new Uint32Array(Math.ceil(networkDims.last / 32)).map(_=>torus.generator() * (2 ** networkDims.last));
targetPattern = target.split('')
>input
>target

// Обучаем сеть (100 итераций)
for (let epoch = 0; epoch < 1; epoch++) {
    deepNet.train(inputPattern, targetPattern);
}

>"Выход сети ПОСЛЕ обучения:"
>result = deepNet.forward(inputPattern).join('');
>target === result;



In [ ]:
pack_to_uint32 = function pack_to_uint32(data) {
    const bits_per_element = Uint32Array.BYTES_PER_ELEMENT *8;
    let out_size = Math.ceil(data.length / bits_per_element);
    let packed_data = new Uint32Array(out_size).fill(0);
    for (let i = 0; i < data.length; i++) {
        let idx = Math.floor(i / bits_per_element);
        let shift = i % bits_per_element;
        packed_data[idx] ^= data[i] << shift;
    }
    return packed_data;
}

In [ ]:
countBigIntOnes = function countBigIntOnes(n) {
    let count = 0;
    while (n > 0n) {
        n &= (n - 1n); // Сбрасываем младшую единицу
        count++;
    }
    return count;
}
nn.BinLinear = class extends nn.Module{
    constructor(net = {}) {
        net = Object.assign({shape_in: 32, shape_out: 32}, net);
        if(!Array.isArray(net.shape_in))
            net.shape_in = [net.shape_in];
        if(!Array.isArray(net.shape_out))
            net.shape_out = [net.shape_out];
        super(net);
    }
    // get model(){
    //     return JSON.stringify(this, (key, value)=>{
    //         if (typeof value === 'bigint')
    //             return value.toString();
    //         return value;
    //     }, 2);
    // }
    // restoreModel(model){
    //     for (let n in model){
    //         const prop = model[n];
    //         this[n] = this.props[n] ??= ((prop)=>{
    //             if(n === 'weights')
    //                 return prop.map(v => BigInt(v))
    //             return prop;
    //         })(prop)
    //     }
    // }
    __init__() {
        this.size_in = this.shape_in.mul();
        this.size_out = this.shape_out.mul();
        let bits_per_element = Uint32Array.BYTES_PER_ELEMENT * 8;
        let shape_in = [...this.shape_in];
        shape_in.last = Math.ceil(shape_in.last / bits_per_element);
        this.weights = tensor.rand([this.shape_out, shape_in], 2**bits_per_element, Uint32Array)._label('weights').p;
        // this.weights = new tensor(Array(this.size_out).fill().map(v=>{
        //     let n = Array(this.size_in).fill().map(_=>Math.round(torus.generator())).join('');
        //     return BigInt('0b'+ n);
        // }), {shape: [this.shape_in, this.shape_out].flat(Infinity)})._label('weights').p;
    }

    forward(input = {data: new Uint32Array(1)}){ // input - Uint32Array
        // let key  = 'BinLinear: '+ dim;
        // let out = torus.get_out(this, key);
        let {data} = input;
        let input_data = data;
        // if(input_data.length){
        //     input_data = input_data.map(v => v > 0 ? 1 : 0).join('').padStart(this.size_in, '0');
        //     input_data = BigInt('0b' + input_data);
        // }
        let res = Array(this.weights.shape[1]);
        res.fill([new Uint32Array(this.weights.shape[0]), new Uint32Array(this.weights.shape[0])]);
        let s_info = this.weights.shape_info;
        for (let i = 0; i < res.length; i++) {
            let stride = i * s_info[1].stride;
            for (let j = 0; j < this.weights.shape[0]; j++) {
                res[i][0][j] = input_data[j] & this.weights.data[stride + j];
                res[i][1][j] = input_data[j] & ~this.weights.data[stride + j];
            }
        }
        // let res = this.weights.data.map(w => [input_data & w, input_data & ~w]);
        
        data = res.map(r => popcountBigIntBuffer32(r[0]) - popcountBigIntBuffer32(r[1]));
        data = data.map(v => v>0? 1: 0);
        // data = res.map(r => countBigIntOnes(r[0]) - countBigIntOnes(r[1]));

        let bits_per_element = Uint32Array.BYTES_PER_ELEMENT * 8;
        let shape_out = [...this.shape_out];
        shape_out.last = Math.ceil(shape_out.last / bits_per_element);
        let out =  tensor.from(data)._src(input, [this.weights])._label('BinLinear')._shape(shape_out);
        // let out =  tensor.from(data)._src(input, [this.weights])._label('BinLinear')._shape(this.shape_out);
        out._back = () => {
            
            // let bin_res = res.map(r=>[r[0].toString(2).padStart(this.size_in, '0').split(''), r[1].toString(2).padStart(this.size_in, '0').split('')]);
            // bin_res = bin_res.map(r=> r[0].map((r0,i)=>{
            //     return +r0 - +r[1][i]
            // }) )
            
            // input_data = input_data.toString(2).padStart(this.size_in, '0').split('').map(v => +v);
            // let weights = this.weights.data.map(w =>w.toString(2).padStart(this.size_in, '0').split('').map(v => +v));
            // let input_T = T(input_data);
            // let weights_T = T(weights);
            // let d_out = out.grad.data;
            // let d_weights = matrixMul(input_T, d_out);
            // let d_input = matrixMul(d_out, weights_T);
            
            // bin_res = bin_res.map(r=>r[0].split('').map((r0) )
            // let grad = out.grad.data.map(v => v > 0 ? 1 : 0).join('').padStart(this.size_out, '0');
            // grad = BigInt('0b' + grad);
            // if (!out.grad.data.some(v => v))
            //     return;
            // console.log(grad);
            // grad = this.weights.data.map((w, i) => );
            // console.log(grad);

            // // let error = data ^ target;
            // // if (error === 0n)
            // //     return;
            // let target = undefined;

            // let input_data = input.data;
            // if (input.data.length) {
            //     input_data = input.data.map(v => v>0 ? 1 : 0).join('').padStart(this.size_in, '0');
            //     input_data = BigInt('0b' + input_data);
            // }
        
            // for (let i = 0n; i < BigInt(this.size_out); i++){
            //     // if (((error >> i) & 1n) === 0n)
            //     if (out.grad.data[i] === 0)
            //         continue;
            //     let error = out.grad.data[i];
            //     let step = (Math.sign(error) || -1) * -2;
            //     let pos, max = this.size_in;
            //     if (step > 0) {
            //         if (error === -1) {
            //             pos = BigInt(Math.trunc(this.size_in * torus.generator()));
            //             if (((input_data >> pos) & 1n) !== ((this.weights.data[i] >> pos) & 1n)) {
            //                 // target = input_data ^ (1n << pos);
            //                 if (input.grad) {
            //                     input.grad.data[pos] = ((input_data >> pos) & 1n) === 0n? -1: 1;
            //                     continue;
            //                 }
            //             }
            //         }
            //         // while (error < 0 && max--) {
            //             pos = BigInt(Math.trunc(this.size_in * torus.generator()));
            //             if (((this.weights.data[i] >> pos) & 1n) === 0n){
            //                 this.weights.data[i]  = this.weights.data[i]  ^ (1n << pos);
            //                 error += step;
            //             }
            //         // }
        
            //         if (error > 0/* 1 */) {
            //             pos = BigInt(Math.trunc(this.size_in * torus.generator()));
            //             if (((input_data >> pos) & 1n) === ((this.weights.data[i] >> pos) & 1n)) {
            //                 // target = input_data ^ (1n << pos);
            //                 if (input.grad)
            //                     input.grad.data[pos] = ((input_data >> pos) & 1n) === 0n? -1: 1;
            //             }
            //         }
            //         else if (error < 0/* 1 */) {
            //             pos = BigInt(Math.trunc(this.size_in * torus.generator()));
            //             if (((input_data >> pos) & 1n) !== ((this.weights.data[i] >> pos) & 1n)) {
            //                 // target = input_data ^ (1n << pos);
            //                 if (input.grad)
            //                     input.grad.data[pos] = ((input_data >> pos) & 1n) === 0n? -1: 1;
            //             }
            //         }
            //     }
            //     else {
            //         if (error === 1) {
            //             pos = BigInt(Math.trunc(this.size_in * torus.generator()));
            //             if (((input_data >> pos) & 1n) === ((this.weights.data[i] >> pos) & 1n)) {
            //                 // target = input_data ^ (1n << pos);
            //                 if (input.grad) {
            //                     input.grad.data[pos] = ((input_data >> pos) & 1n) === 0n? -1: 1;
            //                     continue;
            //                 }
            //             }
            //         }
            //         // while (error > 0 && max--) {
            //             pos = BigInt(Math.trunc(this.size_in * torus.generator()));
            //             if (((this.weights.data[i] >> pos) & 1n) === 1n) {
            //                 this.weights.data[i]  = this.weights.data[i]  ^ (1n << pos);
            //                 error += step;
            //             }
            //         // }
            //         if (error < 0) {
            //             pos = BigInt(Math.trunc(this.size_in * torus.generator()));
            //             if (((input_data >> pos) & 1n) !== ((this.weights.data[i] >> pos) & 1n)) {
            //                  // target = input_data ^ (1n << pos);
            //                 if (input.grad)
            //                     input.grad.data[pos] = ((input_data >> pos) & 1n) === 0n? -1: 1;
            //             }
            //         }
            //         else if (error > 0) {
            //             pos = BigInt(Math.trunc(this.size_in * torus.generator()));
            //             if (((input_data >> pos) & 1n) === ((this.weights.data[i] >> pos) & 1n)) {
            //                 // target = input_data ^ (1n << pos);
            //                 if (input.grad)
            //                     input.grad.data[pos] = ((input_data >> pos) & 1n) === 0n? -1: 1;
            //             }
            //         }
            //     }
            // }
            // input.grad?.data.reverse();
        }
        return out;
    }

}

In [ ]:
out = 111n
target = 12n
error = out^target
>error
out = out.toString(2)
target = target.toString(2)
error = error.toString(2)
max = Math.max(out.length, target.length);
>out.padStart(max, '0');
>target.padStart(max, '0');
>error.padStart(max, '0');


<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>error</label>

99

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>out.padStart(max, '0')</label>

1101111

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>target.padStart(max, '0')</label>

0001100

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>error.padStart(max, '0')</label>

1100011

In [ ]:
val = 49n
>val.toString(2);
for(let i = 0; val; i++){
    let bit = val & 1n;
    val >>= 1n;
    >bit
}



In [ ]:
trans = weights_b.reduce((res, val, idx)=>{
    for(let i = 0n; i<size_in; i++){
        res[i] += val[i]
    }
    return res;
}, Array(size_in).fill('')).map(v=>BigInt('0b' + v))
>trans.map(v => v.toString(2).padStart(size_out, '0'));

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>trans.map(v => v.toString(2).padStart(size_out, '0'))</label>

[
  "1011",
  "1001",
  "1011",
  "1110",
  "0101",
  "1100"
]

In [ ]:
trans = weights.reduce((res, val, idx)=>{
    for(let i = 0n; i<size_in; i++){

        if(val){
            let bit = val & 1n;
            let v = res[i];
            v <<= 1n
            v |= bit    
            res[i] = v
            val >>= 1n;  
        }
        else
            res[i] <<= 1n
            

    }
    return res;
}, Array(size_in).fill(0n))
>trans.map(v => v.toString(2).padStart(size_out, '0'));

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>trans.map(v => v.toString(2).padStart(size_out, '0'))</label>

[
  "1100",
  "0101",
  "1110",
  "1011",
  "1001",
  "1011"
]

In [ ]:
dataset = await this.loadFile('minst-num.json'); 
example = dataset[0][0]


In [ ]:
dataset = await this.loadFile('minst-num.json'); 
example = dataset[0][0]


In [ ]:
MNIST = class MNIST  extends nn.NeuroModule{
    constructor(props = {dim: 28}) {
        super(props);
        
    }    
    __init__(){
        this.layer1 = new nn.BinLinear({shape_in: [this.dim,this.dim], shape_out: [this.dim/2, this.dim/2]})
        this.layer2 = new nn.BinLinear({shape_in: [this.dim/2, this.dim/2], shape_out: 10})
    }    
    forward(x){
        x = this.layer1(x)
        x = this.layer2(x)
        // x = softmaxBin(x);
        return x
    }
}

In [ ]:
net = new MNIST();
model_name = 'mnist2.json';
try{ 
    // net = await this.loadFile(model_name); 
    // net = new MNIST(net);
}
catch(e){ }

// >net.model
this.saveFile(net.model, model_name);
result = net({data: BigInt(example)})
>result.data


<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>result.data</label>

0,-4,-18,-6,-4,-4,-10,-16,4,10

In [ ]:
countBigIntOnes = function(n) {
    let count = 0;
    while (n > 0n) {
        n &= (n - 1n); // Сбрасываем младшую единицу
        count++;
    }
    return count;
}


In [ ]:
loss = 0
count = 100
for(let i = 0; i<count; i++){
    for(let n in dataset){
        list = dataset[n];
        let pos = Math.trunc(list.length * torus.generator());
        let example = list[pos];
        
        let target = BigInt(2 ** n)
        target = new tensor([target], {dType: BigInt, shape: [1, 10]})._label('target')
        example = BigInt(example);
        let result = await net({data: example});
        >result.data
        let loss = result.MAE(target)
        >loss.data
        loss.back()

    }    
}

loss = loss / (count*10)
>loss

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>result.data</label>

3,-7,-15,-13,-1,-19,-11,-17,7,15

In [ ]:
tensor.prototype.binary_loss = function (target = {data: 0n}) {
    target = target.data;
    if (!target.length) {
        target = Array(this.length).fill(0).map((_,i) => (target >> BigInt(i)) & 1n);
    }
    let out = torus.get_out(this, 'binary_loss');
    if (!out) {
        out = tensor.from(new torus.DEFAULT_TYPE(1))._src(this)._label('binary_loss');
        torus.set_out(this, out, 'binary_loss');
        let error = tensor.from(new torus.DEFAULT_TYPE(this.length))._label('binary_loss_error');
        out._fwd = (target) => {
            let loss = 0;
            for (let i = 0; i < this.length; i++) {
                loss += Math.abs(error.data[i] = this.data[i] - target[i % target.length]);
            }
            loss /= this.length;
            out.data.set([loss]);
            return out;
        }
        if (this.allowGrad) {
            out._back = ()=>{
                for (let i = 0; i < this.length; i++)
                    this.grad.data[i] += error.data[i];
            }
        }
    }
    return out._fwd(target);
}

In [ ]:
XOR = class XOR  extends nn.NeuroModule{
    __init__(){
        this.layer1 = new nn.BinLinear({shape_in: 2, shape_out: 2});
        // this.layer2 = new nn.BinLinear({shape_in: 2000, shape_out: 1000});
        this.layer3 = new nn.BinLinear({shape_in: 2, shape_out: 1});
    }    
    async forward(x){
        x = (await this.layer1(x))._label('BinLinear layer1');
        //x = x.ReLU({limit: 1});
        // x = (await this.layer2(x))._label('BinLinear layer2');
        x = (await this.layer3(x))._label('BinLinear layer3');
        x = x.ReLU({limit: 100});
        return x
    }
}

In [ ]:
net = new XOR();
>net.layer1.weights
>net.layer3.weights

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>net.layer1.weights</label>

PARAM (id#2): matrix weights: shape(2,2), length(2), BigInt, 
[[       1,       2]]


<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>net.layer3.weights</label>

PARAM (id#3): matrix weights: shape(2,1), length(1), BigInt, 
[[       1]]


In [ ]:
net = new XOR();
>net.layer1.weights
>net.layer3.weights

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>net.layer1.weights</label>

PARAM (id#680): matrix weights: shape(2,2), length(2), BigInt, 
[[       2,       1]]


<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>net.layer3.weights</label>

PARAM (id#681): matrix weights: shape(2,1), length(1), BigInt, 
[[       2]]


In [ ]:
data = [['00', tensor.from([0])._label('target for 00')],
        ['01', tensor.from([1])._label('target for 01')],
        ['10', tensor.from([1])._label('target for 10')],
        ['11', tensor.from([0])._label('target for 11')]]

In [ ]:
net = new XOR();
let stop = 0;
let count = 0;

while(stop<data.length && count<1){ 
    count++
    stop = 0
    mae_loss = []
    for (let example of data) {
        result = await net({data: BigInt('0b' + example[0])});
        let loss = result.MSE(example[1]);
        >example[0], result, example[1], loss
        loss.back();
        if(!loss.data[0])
            stop++
    }
}

>count


<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>example[0], result, example[1], loss</label>

00

 (id#12): vector ReLU(k=0, limit=100): 1: shape(1), length(1), Float32Array, 
[      0.]


 (id#4): vector target for 00: shape(1), length(1), Float32Array, 
[      0.]


 (id#15): vector mse_loss: shape(1), length(1), Float32Array, 
[      0.]


In [ ]:
net = new XOR();
let stop = 0;
let count = 0;
let equal = false;
let count_array = Array(800);
for(let i=0 ; i < count_array.length; i++) {
    net.layer1.weights.data[0] = 3n;
    net.layer1.weights.data[1] = 3n;
    net.layer3.weights.data[0] = 3n;
    count = stop = 0;
    while(stop<data.length && count<1000){ 
        count++
        stop = 0
        mae_loss = []
        for (let example of data) {
            result = await net({data: BigInt('0b' + example[0])});
            equal = example[1].data[0] === (result.data[0] > 0? 1: 0);
            let loss = result.binary_loss(example[1]) //binary_loss
            mae_loss.push(loss.data[0])
            if (!loss.data[0] || equal) {
                stop++
                continue;
            }
            loss.back();
        }
    //    >mae_loss.toString()
    }
    
    count_array[i] = count
}
>count_array.toString();
>count_array.reduce((r,v) => r+v, 0) / count_array.length

// >count
// //>net.model    
// >net.layer1.weights
// >net.layer3.weights
> 'Обученные веса: [' + net.layer1.weights.data + ']  [' + net.layer3.weights.data  + ']'
/* для MSE среднее число циклов обучения 46
   для MAE среднее число циклов обучения 15
   для binary_loss среднее число циклов обучения 33 */
/* Убрал цикл при изменении веса 
   для MSE среднее число циклов обучения 22
   для MAE среднее число циклов обучения 15
   для binary_loss среднее число циклов обучения 22 */
/* Сейчас корректируется ошибка, а не сырое значение 
   для MSE среднее число циклов обучения 22
   для MAE среднее число циклов обучения 11
   для binary_loss среднее число циклов обучения 11 */
/* Добавил предварительную проверку, что изменить значение необходимо только на 1 или -1 
   для MSE среднее число циклов обучения 23
   для MAE среднее число циклов обучения 12
   для binary_loss среднее число циклов обучения 9.8 */

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>count_array.toString()</label>

7,10,11,16,9,8,5,9,8,7,15,17,22,17,11,13,40,6,6,23,16,12,8,41,8,6,3,19,10,3,11,14,6,14,16,17,4,4,19,8,14,11,4,4,3,18,8,8,7,7,14,3,9,9,3,5,6,16,7,4,8,12,9,3,4,3,18,9,4,3,7,4,3,6,12,4,4,14,7,6,8,5,5,6,4,3,19,6,4,22,16,14,4,3,13,13,26,5,5,12,5,4,11,4,6,4,17,6,9,8,12,8,7,14,20,4,25,13,4,6,7,3,3,3,6,13,13,3,4,8,5,25,7,17,7,3,4,4,7,11,4,3,11,10,3,8,6,5,3,22,20,9,5,5,9,22,24,9,5,27,4,10,8,11,4,7,20,27,9,13,5,6,19,9,20,3,9,13,3,7,8,6,10,8,11,9,9,4,9,6,16,23,4,8,6,9,3,4,6,29,18,3,5,7,5,6,5,4,11,4,7,3,9,4,9,3,10,5,11,3,14,3,10,14,3,9,13,17,4,4,31,11,14,5,11,11,9,6,14,4,3,5,20,8,11,5,20,16,6,3,7,6,10,15,3,4,7,7,4,16,4,5,3,5,9,16,5,4,4,18,15,12,5,5,5,24,5,9,9,4,17,4,15,4,4,11,9,19,14,8,29,14,9,16,4,7,4,12,4,3,3,12,3,15,4,13,4,5,6,5,12,11,32,17,14,10,5,7,7,11,6,23,4,3,11,9,5,4,5,6,3,4,7,20,8,7,4,17,15,4,3,6,4,5,12,4,24,7,22,22,4,3,17,15,3,20,7,5,3,3,4,8,10,4,20,6,20,4,18,17,9,9,6,30,11,5,4,54,18,33,7,8,12,24,3,3,8,7,9,10,8,20,7,20,11,43,35,7,4,7,23,9,14,7,4,8,5,16,15,3,15,8,14,8,11,4,4,14,9,10,5,8,

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>count_array.reduce((r,v) => r+v, 0) / count_array.length</label>

9.6675

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'> 'Обученные веса: [' + net.layer1.weights.data + ']  [' + net.layer3.weights.data  + ']'</label>

Обученные веса: [2,1]  [3]

In [ ]:
XOR3 = class XOR3  extends nn.NeuroModule{
    __init__(){
        this.layer1 = new nn.BinLinear({shape_in: 3, shape_out: 3});
        //this.layer2 = new nn.BinLinear({shape_in: 5, shape_out: 5});
        this.layer3 = new nn.BinLinear({shape_in: 3, shape_out: 1});
    }    
    async forward(x){
        x = await this.layer1(x);
        //x = await this.layer2(x);
        x = await this.layer3(x);
        return x
    }
}

In [ ]:
net3 = new XOR3();

try{ 
    // net3 = await this.loadFile('xor3.json'); 
    // net3 = new XOR(net3);
}
catch(e){ }

this.saveFile(net3.model, 'xor3.json');
//>net.model

In [ ]:
data3 = [['000', tensor.from([0])._label('target for 000')],
         ['001', tensor.from([1])._label('target for 001')],
         ['010', tensor.from([1])._label('target for 010')],
         ['011', tensor.from([0])._label('target for 011')],
         ['100', tensor.from([1])._label('target for 100')],
         ['101', tensor.from([0])._label('target for 101')],
         ['110', tensor.from([0])._label('target for 110')],
         ['111', tensor.from([1])._label('target for 111')]]


In [ ]:
net3 = new XOR3();
let stop = 0;
let count = 0;
let equal = false;

// for (let w1=0n; w1<8n; w1++){
// for (let w2=0n; w2<8n; w2++){
// for (let w3=0n; w3<8n; w3++){
// for (let w4=0n; w4<8n; w4++){
// net3.layer1.weights.data[0] = w1;
// net3.layer1.weights.data[1] = w2;
// net3.layer1.weights.data[2] = w3;
// net3.layer3.weights.data[0] = w4;
count = stop = 0;

net3.layer1.weights.data[0] = 7n;
net3.layer1.weights.data[1] = 7n;
net3.layer1.weights.data[2] = 7n;
net3.layer3.weights.data[0] = 7n;
str = '[' + net3.layer1.weights.data + '] '
if (net3.layer2) str += '[' + net3.layer2.weights.data + '] '
if (net3.layer3) str += '[' + net3.layer3.weights.data + '] '
>str

while(stop<data3.length && count<1000){ 
    count++
    stop = 0
    mae_loss = []
    for (let example of data3) {
        result = await net3({data: BigInt('0b' + example[0])});
        equal = example[1].data[0] === (result.data[0] > 0? 1: 0);
        let loss = result.binary_loss(example[1])
        mae_loss.push(loss.data[0])
        if (!loss.data[0] || equal) {
            stop++
            continue;
        }
        loss.back();
    }
//    >mae_loss.toString()
}

> '' + count + '   ' + stop
//}}}}


// >count
// //>net.model    
// >net.layer1.weights
// >net.layer3.weights
if (count < 100) {
    > 'Обученные веса: [' + net3.layer1.weights.data + ']  [' + net3.layer3.weights.data  + ']'
}

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>str</label>

[0,1,0] [5] 

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'> '' + count + '   ' + stop</label>

100000   4

In [ ]:
ODA({is: 'oda-mnist-input', imports: '@oda/button', template: /*html*/ `
        <style>
            :host { @apply --vertical; }
            .box {margin:1em; width:281px}
            .panel {margin:-1em 1em 1em 1em; width:281px; background: #a9caff}
        </style>

        <svg class='box'  xmlns='http://www.w3.org/2000/svg' viewBox='0 0 281 281' preserveAspectRatio="xMidYMin meet">
            <rect x="0" y="0" width="281" height="281" fill='#4d85cf'/>
            <rect ~for='rects' :x="$for.item?.x" :y="$for.item?.y" width="9" height="9" :fill='$for.item?.v?"#070637":"#a9caff"'
                    @mousemove="_mouseMove($event, $for.item)"/>
        </svg>
        <div horizontal class='panel'>
            <oda-button  icon-size="18" icon="bootstrap:trash3-fill" @tap='input=new Array(28).fill(0).map(_ => new Array(28).fill(0) )'></oda-button>
        </div>
`,
    get rects() {
        return this.input.map((l,y) => l.map((v,x)=> {
            return {x:x*10+1,y:y*10+1,v}
        })).flat()
    },
    input: {
        $def: new Array(28).fill(0).map(_ => new Array(28).fill(0) )
    },
    _mouseMove(x,e) {
        if (!x.buttons) return
        if (e.v === 1) return
        let [i,j] = [Math.floor(e.y/10), Math.floor(e.x/10)]
        this.input[i][j] = 1
    },
    get numInput(){
        return this.input.map(dim=>{
            return parseInt('00'+dim.join('')+'00', 2)
        })
    }
});
input = this.createElement('oda-mnist-input', {})
>input

<label bold onclick='_findCodeEntry(this)' style='text-decoration: underline; padding: 2px; font-size: large; margin-bottom: 4px; cursor: pointer; color: -webkit-link'>input</label>